In [ ]:
%pip install -q ultralytics==8.4.1 pycocotools==2.0.11 tqdm
!git clone https://github.com/mdmohsin212/WasteScope.git

In [ ]:
import sys
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

REPO_DIR = Path("/content/WasteScope")
sys.path.insert(0, str(REPO_DIR))

In [ ]:
import hashlib
import json
import platform
import shutil
import time
import cv2
import torch
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path
from zipfile import ZipFile
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import ultralytics
import yaml
from IPython.display import display
from pycocotools import mask as mask_utils
from pycocotools.coco import COCO
from tqdm.auto import tqdm
from ultralytics import YOLO
from src.evaluation.metrics import coco_results, detection_errors, evaluate_coco


PROJECT_DIR = Path("/content/drive/MyDrive/WasteScope")

RUN_NAME = "S1_yolo26n_seg_960"
CHECKPOINT = "best.pt"
SPLIT = "val"

IMGSZ = None
DEVICE = 0
HALF = True
AP_CONF = 0.001
CONF = 0.25
MATCH_IOU = 0.50
MAX_DET = 100
BOUNDARY_RATIO = 0.02
WARMUP = 10
FORCE_PREDICT = False

ROOT = Path("/content/wastescope")
DATA = ROOT / "segmentation_prepared"
ROOT.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_DIR)
print("GPU:", torch.cuda.get_device_name(DEVICE))
print("Ultralytics:", ultralytics.__version__)

In [ ]:
ANNOTATIONS = DATA / SPLIT / "annotations.json"

if not ANNOTATIONS.is_file():
    archive_path = ROOT / "segmentation_zips" / f"{SPLIT}.zip"
    archive_path.parent.mkdir(parents=True, exist_ok=True)

    if not archive_path.is_file():
        print(f"Copying {SPLIT}.zip from Drive")
        shutil.copy2(
            PROJECT_DIR / "prepared" / "segmentation" / f"{SPLIT}.zip", archive_path
        )

    DATA.mkdir(parents=True, exist_ok=True)
    with ZipFile(archive_path) as archive:
        for member in tqdm(
            archive.infolist(), desc=f"Extracting {SPLIT}"
        ):
            archive.extract(member, DATA)

SOURCE_WEIGHTS = (
    PROJECT_DIR / "runs" / "segment"
    / RUN_NAME / "weights" / CHECKPOINT
)
WEIGHTS = ROOT / "analysis_weights" / RUN_NAME / CHECKPOINT
WEIGHTS.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(SOURCE_WEIGHTS, WEIGHTS)